In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# محلّل التحسين: دالة Qiskit من Q-CTRL Fire Opal
*راجع [مرجع API](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** دوال Qiskit ميزة تجريبية متاحة فقط لمستخدمي IBM Quantum&reg; Premium Plan وFlex Plan وOn-Prem (عبر IBM Quantum Platform API) Plan. هي في مرحلة إصدار معاينة وقابلة للتغيير.


<Accordion>
<AccordionItem title="إصدارات الحزم">

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## نظرة عامة
مع محلّل التحسين Fire Opal، يمكنك حلّ مسائل التحسين على النطاق الاستخدامي على الأجهزة الكمومية دون الحاجة إلى خبرة كمومية. ما عليك سوى إدخال تعريف المسألة رفيع المستوى، ويتولى المحلّل الباقي. سير العمل بالكامل يدرك الضجيج ويستفيد من [إدارة أداء Fire Opal](/guides/q-ctrl-performance-management) في الخلفية. يقدّم المحلّل باستمرار حلولاً دقيقة للمسائل الصعبة كلاسيكيًا، حتى على نطاق الجهاز الكامل لأكبر أجهزة IBM&reg; QPU.

المحلّل مرن ويمكن استخدامه لحلّ مسائل التحسين التوافقية المعرَّفة كدوال هدف أو رسوم بيانية عشوائية. لا يجب ربط المسائل بطبولوجيا الجهاز. المسائل غير المقيدة والمقيدة قابلة للحلّ، بشرط أن تُصاغ القيود كحدود عقوبة. الأمثلة المضمّنة في هذا الدليل توضّح كيفية حلّ مسألة تحسين غير مقيدة ومسألة مقيدة على النطاق الاستخدامي باستخدام أنواع مدخلات مختلفة للمحلّل. المثال الأول يتضمن مسألة max-cut معرَّفة على رسم بياني منتظم ثلاثي 156 عقدة، بينما يعالج المثال الثاني مسألة Minimum Vertex Cover من 50 عقدة معرَّفة بدالة تكلفة.

للحصول على إمكانية الوصول إلى محلّل التحسين، [تواصل مع Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).

## وصف الدالة
يحسّن المحلّل الخوارزمية بالكامل ويؤتمتها، من قمع الأخطاء على مستوى الأجهزة إلى تعيين المسائل بكفاءة والتحسين الكلاسيكي في حلقة مغلقة. خلف الكواليس، تقلّل خطوط أنابيب المحلّل الأخطاء في كل مرحلة، مما يُمكّن الأداء المحسّن المطلوب للتوسع الفعلي. سير العمل الأساسي مستوحى من خوارزمية التحسين الكمومي التقريبي (QAOA)، وهي خوارزمية هجينة كمومية-كلاسيكية. للاطلاع على ملخص تفصيلي لسير عمل محلّل التحسين الكامل، راجع [المخطوطة المنشورة](https://arxiv.org/abs/2406.01743).

![تصوّر لسير عمل محلّل التحسين](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

لحلّ مسألة عامة مع محلّل التحسين:
1. عرِّف مسألتك كدالة هدف، أو رسم بياني، أو سلسلة دوران `SparsePauliOp`.
2. اتصل بالدالة عبر كتالوج دوال Qiskit.
3. شغّل المسألة مع المحلّل واسترجع النتائج.

### صيغ المسائل المقبولة
- تمثيل التعبير متعدد الحدود لدالة هدف. يُنصح بإنشائه في Python بكائن SymPy Poly موجود وتنسيقه كسلسلة نصية باستخدام [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- تمثيل رسم بياني لنوع مسألة محدد. يجب إنشاء الرسم البياني باستخدام مكتبة networkx في Python، ثم تحويله إلى سلسلة نصية باستخدام دالة networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- تمثيل سلسلة الدوران لمسألة محددة. يجب تمثيل سلسلة الدوران ككائن `SparsePauliOp`؛ راجع [التوثيق](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) لمزيد من التفاصيل.

### الخلفيات المدعومة
شغّل الكود التالي لرؤية قائمة الخلفيات المدعومة حاليًا. إذا لم تكن جهازك مدرجًا، [تواصل مع Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) لإضافة الدعم.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

service.backends()

[<IBMBackend('ibm_boston')>,
 <IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_marrakesh')>,
 <IBMBackend('ibm_kingston')>,
 <IBMBackend('ibm_miami')>]

## Benchmarks

[Published benchmarking results](https://arxiv.org/abs/2406.01743) show that the Solver successfully solves problems with over 120 qubits, even outperforming previously published results on quantum annealing and trapped-ion devices. The following benchmark metrics provide a rough indication of the accuracy and scaling of problem types based on a few examples. Actual metrics may differ based on various problem features, such as the number of terms in the objective function (density) and their locality, number of variables, and polynomial order.

The "Number of qubits" indicated is not a hard limitation but represents rough thresholds where you can expect extremely consistent solution accuracy. Larger problem sizes have been successfully solved, and testing beyond these limits is encouraged.

Arbitrary qubit connectivity is supported across all problem types.

| Problem type    | Number of qubits | Example | Accuracy | Total time (s) | Runtime usage (s) | Number of iterations
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Sparsely-connected quadratic problems  | 156 | 3-regular max-cut| 100%     | 1764     | 293          | 16 |
| Higher-order binary optimization | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| Densely-connected quadratic problems | 50 | Fully-connected max-cut| 100%      |  1758    | 268  | 12 |
| Constrained problem with penalty terms | 50 | Weighted Minimum Vertex Cover with 8% edge density | 100%      | 1074     | 215 | 10 |

## Get started

First, authenticate using your [IBM Quantum API key](http://quantum.cloud.ibm.com/). Then, select the Qiskit Function as follows. (This snippet assumes you've already [saved your account](/docs/guides/functions#install-qiskit-functions-catalog-client) to your local environment.)

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

## Example: Unconstrained optimization
Run the [maximum cut](https://en.wikipedia.org/wiki/Maximum_cut) (max-cut) problem. The following example demonstrates the Solver's capabilities on a 156-node, 3-regular unweighted graph max-cut problem, but you can also solve weighted graph problems.

In addition to `qiskit-ibm-catalog`, you will also use the following packages to run this example: `networkx` and `numpy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [3]:
# %pip install networkx numpy

## المعايير المرجعية
تُظهر [نتائج المعايير المرجعية المنشورة](https://arxiv.org/abs/2406.01743) أن المحلّل ينجح في حلّ مسائل تتجاوز 120 كيوبت، متفوقًا حتى على النتائج المنشورة سابقًا على أجهزة التليين الكمومي والأيونات المحاصرة. مقاييس المعايير المرجعية التالية تعطي مؤشرًا تقريبيًا على دقة أنواع المسائل وقابليتها للتوسع بناءً على بعض الأمثلة. قد تختلف المقاييس الفعلية بناءً على خصائص المسألة المختلفة، مثل عدد الحدود في دالة الهدف (الكثافة) وموقعها، وعدد المتغيرات، والرتبة متعددة الحدود.

"عدد الكيوبتات" المشار إليه ليس حدًا صارمًا بل يمثّل عتبات تقريبية حيث يمكن توقع دقة حلول عالية الاتساق. مسائل أكبر حجمًا تمّ حلّها بنجاح، ويُشجّع الاختبار خارج هذه الحدود.

الاتصالية العشوائية بين الكيوبتات مدعومة عبر جميع أنواع المسائل.

| نوع المسألة    | عدد الكيوبتات | مثال | الدقة | الوقت الكلي (ث) | استخدام وقت التشغيل (ث) | عدد التكرارات
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| مسائل تربيعية متفرقة الاتصال  | 156 | max-cut ثلاثي منتظم| 100%     | 1764     | 293          | 16 |
| تحسين ثنائي ذو رتبة عليا | 156 | نموذج Ising spin-glass | 100%      | 1461     | 272          | 16 |
| مسائل تربيعية كثيفة الاتصال | 50 | max-cut متكامل الاتصال| 100%      |  1758    | 268  | 12 |
| مسألة مقيدة بحدود عقوبة | 50 | Minimum Vertex Cover موزون بكثافة حواف 8% | 100%      | 1074     | 215 | 10 |

## البدء
أولًا، سجّل الدخول باستخدام [مفتاح IBM Quantum API](http://quantum.cloud.ibm.com/). ثم اختر دالة Qiskit كما يلي. (هذا المقتطف يفترض أنك [حفظت حسابك](/guides/functions#install-qiskit-functions-catalog-client) في بيئتك المحلية مسبقًا.)

In [4]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [5]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.svg" alt="Output of the previous code cell" />

### 1. تعريف المسألة
يمكنك تشغيل مسألة Max-Cut بتعريف مسألة رسم بياني وتحديد `problem_type='maxcut'`.

In [6]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [8]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 2. تشغيل المسألة
عند استخدام طريقة الإدخال القائمة على الرسم البياني، حدّد نوع المسألة.

In [9]:
# Get job status
print(maxcut_job.status())

QUEUED


### 3. Retrieve the result
Retrieve the optimal cut value from the results dictionary.

<Admonition type="note">
   The mapping of the variables to the bitstring may have changed. The output dictionary contains a `variables_to_bitstring_index_map` sub-dictionary, which helps to verify the ordering.
</Admonition>

In [10]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


تحقق من [حالة](/guides/functions#check-job-status) حمل عمل دالة Qiskit أو استرجع [النتائج](/guides/functions#retrieve-results) كما يلي:

In [11]:
# %pip install numpy networkx sympy

### 1. Define the problem
Define a random MVC problem by generating a graph with randomly weighted nodes.

In [12]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg" alt="Output of the previous code cell" />

### 3. استرجاع النتيجة
استرجع قيمة القطع المثلى من قاموس النتائج.

> **Note:** ربما تغيّر تعيين المتغيرات إلى السلسلة الثنائية. يحتوي قاموس الإخراج على قاموس فرعي `variables_to_bitstring_index_map` يساعد في التحقق من الترتيب.

In [13]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

Now every edge in the graph should include at least one end point from the cover, which can be expressed as the inequality:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Any case where an edge is not connected to the vertex of cover must be penalized. This can be represented in the cost function by adding a penalty of the form $P(1-n_i-n_j+n_i n_j)$ where $P$ is a positive penalty constant. Thus, an unconstrained alternative to the constrained inequality for weighted MVC is:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [14]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

يمكنك التحقق من دقة النتيجة بحلّ المسألة كلاسيكيًا باستخدام محلّلات مفتوحة المصدر مثل [PuLP](https://coin-or.github.io/pulp/) إذا لم يكن الرسم البياني كثيف الاتصال. قد تتطلب مسائل الكثافة العالية محلّلات كلاسيكية متقدمة للتحقق من صحة الحلّ.

## مثال: تحسين مقيد
مثال Max-Cut السابق هو مسألة تحسين ثنائي تربيعي غير مقيد شائعة. يمكن استخدام محلّل التحسين من Q-CTRL لأنواع مسائل متعددة، بما في ذلك التحسين المقيد. يمكنك حلّ أنواع مسائل عشوائية بإدخال تعريف المسألة ممثَّلًا كمتعدد حدود حيث تُنمذَج القيود كحدود عقوبة.

يوضّح المثال التالي كيفية بناء دالة تكلفة لمسألة تحسين مقيدة، [الغطاء الأدنى للرؤوس](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
بالإضافة إلى حزمتي `qiskit-ibm-catalog` و`qiskit`، ستحتاج أيضًا إلى الحزم التالية لتشغيل هذا المثال: `numpy` و`networkx` و`sympy`. يمكنك تثبيت هذه الحزم بإلغاء تعليق الخلية التالية إذا كنت تشغّل هذا المثال في notebook باستخدام نواة IPython.

In [15]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 1. تعريف المسألة
عرِّف مسألة MVC عشوائية بتوليد رسم بياني بعقد موزونة عشوائيًا.

In [16]:
print(mvc_job.status())

QUEUED


![Output of the previous code cell](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

يمكن صياغة نموذج التحسين القياسي للـMVC الموزون كما يلي. أولًا، يجب إضافة عقوبة لأي حالة لا تكون فيها حافة متصلة برأس في المجموعة الفرعية. لذا، ليكن $n_i = 1$ إذا كان الرأس $i$ في الغطاء (أي في المجموعة الفرعية) و$n_i = 0$ في الحالة المعاكسة. ثانيًا، الهدف هو تقليل العدد الكلي للرؤوس في المجموعة الفرعية، ويمكن تمثيله بالدالة التالية:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [17]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


الآن يجب أن تشمل كل حافة في الرسم البياني على الأقل نقطة نهاية واحدة من الغطاء، ويمكن التعبير عن ذلك بالمتراجحة:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

يجب معاقبة أي حالة لا تكون فيها الحافة متصلة برأس من الغطاء. يمكن تمثيل ذلك في دالة التكلفة بإضافة عقوبة من الشكل $P(1-n_i-n_j+n_i n_j)$ حيث $P$ ثابت عقوبة موجب. وبذلك، البديل غير المقيد للمتراجحة المقيدة للـMVC الموزون هو:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$